In [3]:
from typing import TypedDict,Annotated

from langchain_core.messages import BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain.messages import HumanMessage, AIMessage, ToolMessage
from rich import print as rprint

## reducer

In [10]:
def reducer_fun(left: list[BaseMessage], right: list[BaseMessage]) -> list[BaseMessage]:
    return left + right

class MyState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

def tool_node(state: MyState):
    return {"messages": [ToolMessage(content="来自tool_node的内容", tool_call_id="tool_node")]}

def llm_node(state: MyState):
    return {"messages": [AIMessage(content="来自llm_node的内容")]}

builder = StateGraph(state_schema=MyState)

builder.add_node(tool_node)
builder.add_node(llm_node)
builder.add_edge(START, "tool_node")
builder.add_edge("tool_node", "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()

res = graph.invoke({"messages": [HumanMessage(content="你是一个AI助手")]})

rprint(res)

{
    'messages': [
        HumanMessage(
            content='你是一个AI助手',
            additional_kwargs={},
            response_metadata={},
            id='ac206bed-ce53-411d-ad42-2a2243bbe5bf'
        ),
        ToolMessage(content='来自tool_node的内容', id='1', tool_call_id='tool_node'),
        AIMessage(
            content='来自llm_node的内容',
            additional_kwargs={},
            response_metadata={},
            id='d823b78b-76f1-428d-853f-a658ce9bb6e8',
            tool_calls=[],
            invalid_tool_calls=[]
        )
    ]
}

## 四种状态

In [11]:
class InputSchema(TypedDict):
    username: str

class OutputSchema(TypedDict):
    graph_output: str

class OverAllSchema(TypedDict):
    nickname: str
    username: str
    graph_output: str

class PrivateSchema(TypedDict):
    greeting: str

def node_1(state: InputSchema) -> OverAllSchema:
    return {
        "nickname": "Dear " + state["username"]
    }

def node_2(state: OverAllSchema) -> PrivateSchema:
    return {
        "greeting": state["nickname"] + ", 早上好"
    }

def node_3(state: PrivateSchema) -> OutputSchema:
    return {
        "graph_output": state["greeting"] + ", 很高兴见到你"
    }

builder = StateGraph(
    state_schema=OverAllSchema,
    input_schema=InputSchema,
    output_schema=OutputSchema
)

builder.add_node(node_1)
builder.add_node(node_2)
builder.add_node(node_3)

builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

graph = builder.compile()

res = graph.invoke({"username": "atguigu"})

print(res)

{'graph_output': 'Dear atguigu, 早上好, 很高兴见到你'}
